# Importance Analysis

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from thesis_project import (
    config as global_config,
)

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = "importance-analysis"

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

# Main

## Importance Mutual Information

In [ ]:
import datetime


def float_to_time(x):
    return str(datetime.timedelta(seconds=int(x)))


print(float_to_time(65532.52))

In [ ]:
from natsort import natsorted

TICKER = "fbtp"

# run 1 and 2 are preliminary
runs = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

partial = {}
for run in runs:
    partial[run] = pd.read_parquet(
        global_config.INT_DIR
        / TICKER
        / "importance-mi"
        / "runs"
        / f"run_{run:03d}"
        / "results.parquet"
    )


results = pd.concat(partial.values(), ignore_index=True)

pairs = results[["horizon", "target"]].drop_duplicates().values
pairs = natsorted(pairs, key=lambda x: (x[1], x[0]))
n = 20
key = "cmi"
print(f"Top {n} features by {key} for each horizon and target:\n")
for h, t in pairs:
    mask = (results["horizon"] == h) & (results["target"] == t)
    data = results[mask].sort_values([key], ascending=False)
    data = data.drop(columns=["horizon", "target", "n_samples", "timestamp"])
    data = data.reset_index(drop=True)
    data.index += 1
    data.index.name = "rank"

    title = f"Top {n} features by {key} for horizon {h}s and target {t}"
    # data = data.head(n)
    mask = data["feature"].str.contains("aof_lvl")
    data = data[mask]
    display(data.style.set_caption(title))
    # data.to_csv(OUT_TAB_DIR / f"{t}_top_{n}_features_vs_by_{key}_horizon_{h}s.csv")